# LeetCode 750 — Number Of Corner Rectangles bằng Python

Link đề: https://leetcode.com/problems/number-of-corner-rectangles/ · Độ khó: **Medium** · Chủ đề: Array, Math, Dynamic Programming, Matrix, Hash Table

> **Bài Premium (🔒).** Đề trong notebook này được viết theo hiểu biết về bài (kèm các ví dụ trong tài liệu tham khảo); nếu bạn có tài khoản Premium, hãy đối chiếu lại đề gốc trên LeetCode.

Skeleton LeetCode:

```python
class Solution:
    def countCornerRectangles(self, grid: List[List[int]]) -> int:
        ...
```

Ghi chú: bài này không cần cấu trúc tự dựng. Mọi lời giải được đối chiếu với một oracle làm đúng từng chữ theo định nghĩa (duyệt mọi bộ bốn góc) trên lưới nhỏ, kể cả **vét cạn mọi lưới nhị phân** cỡ nhỏ.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: đếm số **hình chữ nhật góc** trên lưới `0/1`, tức bốn ô `1` phân biệt là bốn đỉnh của một hình chữ nhật song song các trục (chỉ bốn góc cần bằng `1`, bên trong tùy ý).
2. Nắm các công cụ Python: truy cập lưới `grid[r][c]`, tổ hợp `math.comb`, `dict`/`Counter` với khóa là tuple, số nguyên làm **bitmask** (`&`, `<<`, `int.bit_count()`), `itertools.combinations`, và đo thời gian bằng `time.perf_counter`.
3. Đi từ **vét cạn bốn vòng lặp** (`O(m²n²)`) → **đếm cột chung của từng cặp hàng rồi dùng `C(k, 2)`** (`O(m²n)`) → **bitmask + popcount** (`O(m² · n/64)`) → **bảng băm đếm cặp cột** (`O(m·n²)`, lời giải nộp; thực tế chỉ phụ thuộc số ô `1`) → **đảo vai trò: đếm theo cặp hàng của từng cột**.
4. Hiểu mấu chốt: hai hàng có đúng `k` cột cùng bằng `1` tạo ra `C(k, 2)` hình chữ nhật (chọn 2 trong `k` cột chung); tổng theo mọi cặp hàng là đáp án. Không cần liệt kê bộ bốn góc.
5. Tránh các lỗi kinh điển: cộng `k` thay vì `C(k, 2)`, đếm cặp hàng hai lần (cả `(i, j)` và `(j, i)`), cập nhật bảng băm **trước** khi cộng, quên rằng bốn góc phải phân biệt (một hàng đơn không tạo được hình chữ nhật), và dùng cách `O(m²n²)` làm quá thời gian.

## 2. Đề bài & ví dụ

Cho một lưới `m x n` gồm các số `0` hoặc `1`. Hãy trả về **số hình chữ nhật góc** (corner rectangles).

Một **hình chữ nhật góc** là bốn ô `1` **phân biệt** trên lưới tạo thành bốn đỉnh của một hình chữ nhật song song các trục tọa độ. Chú ý chỉ **bốn góc** cần bằng `1`; các ô bên trong và trên cạnh có thể là `0` hoặc `1`. Bốn ô `1` dùng làm góc phải **khác nhau** (nên hình chữ nhật có ít nhất hai hàng và hai cột khác nhau).

**Ví dụ 1**

- Input: `grid = [[1,0,0,1,0],[0,0,1,0,1],[0,0,0,1,0],[1,0,1,0,1]]`
- Output: `1`
- Giải thích: chỉ có một hình chữ nhật góc, với bốn góc `grid[1][2]`, `grid[1][4]`, `grid[3][2]`, `grid[3][4]`.

**Ví dụ 2**

- Input: `grid = [[1,1,1],[1,1,1],[1,1,1]]`
- Output: `9`
- Giải thích: có bốn hình `2x2`, bốn hình `2x3` và `3x2`, và một hình `3x3`: tổng `9`.

**Ví dụ 3**

- Input: `grid = [[1,1,1,1]]`
- Output: `0`
- Giải thích: hình chữ nhật cần bốn góc phân biệt (cần ít nhất hai hàng).

**Ràng buộc:**

- `m == grid.length`, `n == grid[i].length`.
- `1 <= m, n <= 200`.
- `grid[i][j]` là `0` hoặc `1`.
- Số ô `1` trong lưới nằm trong khoảng `[1, 6000]`.

Chú ý: `200 x 200 = 40000` ô, nhưng chỉ có tối đa `6000` ô `1`. Ràng buộc này rất quan trọng cho việc chọn thuật toán.

In [1]:
from typing import List
from collections import Counter
from itertools import combinations
from math import comb
import random
import time

examples = [
    ([[1, 0, 0, 1, 0], [0, 0, 1, 0, 1], [0, 0, 0, 1, 0], [1, 0, 1, 0, 1]], 1),
    ([[1, 1, 1], [1, 1, 1], [1, 1, 1]], 9),
    ([[1, 1, 1, 1]], 0),
]

def show_grid(grid):
    for row in grid:
        print(" ".join(str(x) for x in row))

for grid, expected in examples:
    print(f"m={len(grid)}, n={len(grid[0])}, số ô 1 = {sum(map(sum, grid))} -> kỳ vọng {expected}")
show_grid(examples[0][0])

m=4, n=5, số ô 1 = 8 -> kỳ vọng 1
m=3, n=3, số ô 1 = 9 -> kỳ vọng 9
m=1, n=4, số ô 1 = 4 -> kỳ vọng 0
1 0 0 1 0
0 0 1 0 1
0 0 0 1 0
1 0 1 0 1


## 3. Kiến thức nền

### 3.1. Lưới hai chiều: `grid[r][c]`, `len(grid)`, `len(grid[0])`

Lưới là list của các list: `grid[r]` là hàng thứ `r`, `grid[r][c]` là ô ở hàng `r`, cột `c`. `m = len(grid)` là số hàng, `n = len(grid[0])` là số cột (vì mọi hàng cùng độ dài). Chỉ số bắt đầu từ `0`.

In [2]:
grid = examples[0][0]
m, n = len(grid), len(grid[0])
print("m =", m, "| n =", n)
print("grid[1] =", grid[1], "| grid[1][2] =", grid[1][2], "| grid[3][4] =", grid[3][4])
print("các cột có 1 ở hàng 3:", [c for c in range(n) if grid[3][c]])

m = 4 | n = 5
grid[1] = [0, 0, 1, 0, 1] | grid[1][2] = 1 | grid[3][4] = 1
các cột có 1 ở hàng 3: [0, 2, 4]


### 3.2. Tổ hợp `C(k, 2)` và `math.comb`

`C(k, 2) = k·(k-1)/2` là số cách chọn **2 phần tử** trong `k` phần tử (không phân biệt thứ tự). Nếu hai hàng cùng có `1` ở `k` cột, mỗi cách chọn 2 trong `k` cột đó cho đúng một hình chữ nhật. Ví dụ `k = 3` cột chung cho `C(3, 2) = 3` hình chữ nhật. Python có `math.comb(k, 2)`; `comb(1, 2) = 0` và `comb(0, 2) = 0`, nên không cần xử lý riêng `k < 2`.

In [3]:
for k in range(0, 6):
    print(f"k={k}: C(k,2) = {comb(k, 2)} = k*(k-1)//2 = {k * (k - 1) // 2}")
print("các cặp cột trong {0,2,5}:", list(combinations([0, 2, 5], 2)))

k=0: C(k,2) = 0 = k*(k-1)//2 = 0
k=1: C(k,2) = 0 = k*(k-1)//2 = 0
k=2: C(k,2) = 1 = k*(k-1)//2 = 1
k=3: C(k,2) = 3 = k*(k-1)//2 = 3
k=4: C(k,2) = 6 = k*(k-1)//2 = 6
k=5: C(k,2) = 10 = k*(k-1)//2 = 10
các cặp cột trong {0,2,5}: [(0, 2), (0, 5), (2, 5)]


### 3.3. `dict` / `Counter` với khóa là tuple

Tuple `(i, j)` bất biến và băm được, nên dùng làm khóa của `dict`. `Counter` là `dict` chuyên đếm: khóa vắng mặt cho giá trị `0` (không báo lỗi), và `cnt[key] += 1` tăng bộ đếm. Ta sẽ dùng nó để nhớ "đã có bao nhiêu hàng trước đó cùng có `1` ở cặp cột `(i, j)`".

In [4]:
cnt = Counter()
print("chưa có:", cnt[(0, 3)])
cnt[(0, 3)] += 1
cnt[(0, 3)] += 1
cnt[(2, 4)] += 1
print(cnt, "| cnt[(0, 3)] =", cnt[(0, 3)], "| cnt[(1, 1)] =", cnt[(1, 1)])

chưa có: 0
Counter({(0, 3): 2, (2, 4): 1}) | cnt[(0, 3)] = 2 | cnt[(1, 1)] = 0


### 3.4. Bitmask: một hàng của lưới thành một số nguyên

Mỗi hàng `0/1` có thể nén vào **một số nguyên**: bit thứ `j` của số bằng `grid[r][j]`. Khi đó **phép AND `&`** của hai hàng cho đúng các cột mà **cả hai** cùng có `1`, và `int.bit_count()` (Python 3.10+) đếm số bit `1` nhanh bằng phần cứng. Python cho số nguyên dài tùy ý, nên hàng 200 cột vẫn là một số nguyên.

In [5]:
def row_mask(row):
    mask = 0
    for j, v in enumerate(row):
        if v:
            mask |= 1 << j
    return mask

a, b = [1, 0, 1, 1, 0], [1, 1, 1, 0, 0]
ma, mb = row_mask(a), row_mask(b)
print(f"{ma:05b} & {mb:05b} = {ma & mb:05b} (đọc từ phải sang trái là cột 0,1,2,...) -> số cột chung = {(ma & mb).bit_count()}")
print("cột chung thật sự:", [j for j in range(5) if a[j] and b[j]])

01101 & 00111 = 00101 (đọc từ phải sang trái là cột 0,1,2,...) -> số cột chung = 2
cột chung thật sự: [0, 2]


### 3.5. `itertools.combinations` để liệt kê cặp

`combinations(iterable, 2)` sinh mọi cặp `(x, y)` với `x` đứng trước `y` trong dãy (không lặp, không đổi chỗ). Ta dùng nó cho cặp hàng `(r1, r2)` hoặc cặp cột `(c1, c2)` khi viết bản oracle.

In [6]:
print(list(combinations(range(4), 2)))
print("số cặp của 200 hàng:", comb(200, 2), "| số bộ (cặp hàng x cặp cột) tối đa:", comb(200, 2) * comb(200, 2))

[(0, 1), (0, 2), (0, 3), (1, 2), (1, 3), (2, 3)]
số cặp của 200 hàng: 19900 | số bộ (cặp hàng x cặp cột) tối đa: 396010000


### 3.6. Đo thời gian với `time.perf_counter`

Để thấy tận mắt khác biệt giữa các cách, ta đo thời gian. `time.perf_counter()` trả về giây dưới dạng số thực; hàm `timeit_once` lấy hiệu hai lần đo.

In [7]:
def timeit_once(fn, *args):
    t0 = time.perf_counter()
    res = fn(*args)
    return res, time.perf_counter() - t0

res, dt = timeit_once(sum, range(10 ** 6))
print("sum =", res, "| khoảng", round(dt * 1000, 2), "ms")

sum = 499999500000 | khoảng 23.53 ms


## 4. Ý tưởng nền tảng / trực giác

Vẽ ví dụ 1:

```
cột:  0 1 2 3 4
h0:   1 0 0 1 0
h1:   0 0 1 0 1
h2:   0 0 0 1 0
h3:   1 0 1 0 1
```

Một hình chữ nhật góc có **hai hàng** `r1 < r2` và **hai cột** `c1 < c2` sao cho bốn ô `(r1,c1), (r1,c2), (r2,c1), (r2,c2)` đều là `1`. Hãy nhìn theo **cặp hàng**: với hai hàng bất kỳ, các cột mà **cả hai** đều có `1` là "cột chung". Hai cột chung bất kỳ cho một hình chữ nhật.

Tính tay các cặp hàng của ví dụ 1 (tập cột có `1` của mỗi hàng: `h0 = {0,3}`, `h1 = {2,4}`, `h2 = {3}`, `h3 = {0,2,4}`):

| cặp hàng | cột chung | số cột chung `k` | `C(k, 2)` |
|---|---|---|---|
| (h0, h1) | {} | 0 | 0 |
| (h0, h2) | {3} | 1 | 0 |
| (h0, h3) | {0} | 1 | 0 |
| (h1, h2) | {} | 0 | 0 |
| (h1, h3) | {2, 4} | 2 | **1** |
| (h2, h3) | {} | 0 | 0 |

Tổng `= 1`. Khớp đáp án! Với lưới toàn `1` cỡ `3x3`: có `C(3,2) = 3` cặp hàng, mỗi cặp có `3` cột chung cho `C(3,2) = 3` hình, tổng `9`. Khớp ví dụ 2.

Vậy **đáp án = tổng, theo mọi cặp hàng, của `C(số cột chung, 2)`**. Hãy kiểm chứng bằng code.

In [8]:
def common_columns(grid, r1, r2):
    return [c for c in range(len(grid[0])) if grid[r1][c] and grid[r2][c]]

for grid, expected in examples:
    total = 0
    for r1, r2 in combinations(range(len(grid)), 2):
        common = common_columns(grid, r1, r2)
        total += comb(len(common), 2)
        if len(grid) <= 4:
            print(f"cặp hàng ({r1},{r2}): cột chung {common} -> C({len(common)},2) = {comb(len(common), 2)}")
    print("tổng =", total, "| kỳ vọng", expected)
    assert total == expected

cặp hàng (0,1): cột chung [] -> C(0,2) = 0
cặp hàng (0,2): cột chung [3] -> C(1,2) = 0
cặp hàng (0,3): cột chung [0] -> C(1,2) = 0
cặp hàng (1,2): cột chung [] -> C(0,2) = 0
cặp hàng (1,3): cột chung [2, 4] -> C(2,2) = 1
cặp hàng (2,3): cột chung [] -> C(0,2) = 0
tổng = 1 | kỳ vọng 1
cặp hàng (0,1): cột chung [0, 1, 2] -> C(3,2) = 3
cặp hàng (0,2): cột chung [0, 1, 2] -> C(3,2) = 3
cặp hàng (1,2): cột chung [0, 1, 2] -> C(3,2) = 3
tổng = 9 | kỳ vọng 9
tổng = 0 | kỳ vọng 0


## 5. Cách 1 — Brute Force: bốn vòng lặp

### Ý tưởng

Làm đúng theo định nghĩa: duyệt mọi cặp hàng `r1 < r2` và mọi cặp cột `c1 < c2`, đếm khi bốn góc đều bằng `1`.

### Code

In [9]:
def count_brute(grid):
    m, n = len(grid), len(grid[0])
    ans = 0
    for r1 in range(m):
        for r2 in range(r1 + 1, m):
            for c1 in range(n):
                for c2 in range(c1 + 1, n):
                    if grid[r1][c1] and grid[r1][c2] and grid[r2][c1] and grid[r2][c2]:
                        ans += 1
    return ans

for grid, expected in examples:
    got = count_brute(grid)
    print(f"m={len(grid)}, n={len(grid[0])} -> {got}")
    assert got == expected

m=4, n=5 -> 1
m=3, n=3 -> 9
m=1, n=4 -> 0


### Độ phức tạp và nút thắt

- **Time:** `C(m, 2) · C(n, 2)` bộ bốn góc, tức `O(m² n²)`. Với `m = n = 200` đó là cỡ `4 · 10^8` (gần `4·10^8` lần kiểm tra), quá chậm trong Python.
- **Space:** `O(1)`.

Nút thắt: ta xét cả những bộ có góc bằng `0`. Cách đo thời gian bên dưới cho thấy thời gian tăng với lũy thừa bốn của kích thước lưới.

In [10]:
def random_grid(rng, m, n, p):
    return [[1 if rng.random() < p else 0 for _ in range(n)] for _ in range(m)]

rng = random.Random(1)
for size in (10, 20, 40):
    g = random_grid(rng, size, size, 0.3)
    res, dt = timeit_once(count_brute, g)
    print(f"lưới {size}x{size}: đáp án {res:6d}, thời gian {dt * 1000:8.1f} ms")
print("gấp đôi kích thước -> thời gian gấp ~16 lần (O(m^2 n^2)); 200x200 sẽ lâu gấp hàng nghìn lần 40x40")

lưới 10x10: đáp án      6, thời gian      0.1 ms
lưới 20x20: đáp án    476, thời gian      1.5 ms
lưới 40x40: đáp án   3766, thời gian     24.9 ms
gấp đôi kích thước -> thời gian gấp ~16 lần (O(m^2 n^2)); 200x200 sẽ lâu gấp hàng nghìn lần 40x40


## 6. Ý tưởng thuật toán / bước đột phá

**Bước đột phá:** thay vì chọn bốn góc, ta chọn **hai hàng** rồi **đếm** số cột chung `k`. Mọi cặp cột trong `k` cột chung đều tạo ra một hình chữ nhật, nên hai hàng đó góp đúng `C(k, 2) = k(k-1)/2` hình. Ta không cần liệt kê từng hình.

Điều này đưa độ phức tạp từ `O(m² n²)` xuống `O(m² n)`.

Một góc nhìn khác (đối xứng): thay vì cặp hàng, hãy theo dõi **cặp cột**. Khi xử lý lần lượt từng hàng, mỗi cặp cột `(i, j)` mà hàng hiện tại có `1` cả hai thì "ghép được" với mọi hàng **trước đó** cũng có `1` ở cả `i` và `j`. Nếu ta nhớ `cnt[(i, j)]` = số hàng trước đó có `1` ở cả hai cột, thì hàng hiện tại thêm đúng `cnt[(i, j)]` hình chữ nhật cho cặp cột này. Sau đó `cnt[(i, j)] += 1`. Cách này chỉ xét các cặp cột **có cả hai bằng `1`**, nên số bước chỉ phụ thuộc số ô `1`: ở hàng có `k` ô `1` ta xét `C(k, 2)` cặp.

Hãy xem hai góc nhìn cho cùng một kết quả trên ví dụ 1, và đo "công việc" của chúng.

In [11]:
def work_row_pairs(grid):
    m, n = len(grid), len(grid[0])
    return comb(m, 2) * n                      # mỗi cặp hàng quét n cột

def work_hash_pairs(grid):
    return sum(comb(sum(row), 2) for row in grid)   # mỗi hàng xét C(k,2) cặp cột có 1

g = examples[0][0]
print("ví dụ 1: cặp hàng quét", work_row_pairs(g), "ô | cặp cột có 1 cả hai:", work_hash_pairs(g))
rng = random.Random(2)
g = random_grid(rng, 200, 200, 0.15)
print(f"lưới 200x200, {sum(map(sum, g))} ô 1: cặp hàng quét {work_row_pairs(g)} | cặp cột có 1: {work_hash_pairs(g)}")

ví dụ 1: cặp hàng quét 30 ô | cặp cột có 1 cả hai: 5
lưới 200x200, 5999 ô 1: cặp hàng quét 3980000 | cặp cột có 1: 89720


## 7. Các cách tiếp theo

Tổng cộng 5 cách:

1. Bốn vòng lặp (đã làm).
2. **Từng cặp hàng, đếm cột chung `k`, cộng `C(k, 2)`** (`O(m²n)`).
3. **Bitmask + popcount**: cùng ý tưởng cách 2 nhưng "đếm cột chung" bằng `&` và `bit_count`.
4. **Bảng băm đếm cặp cột** (lời giải nộp).
5. **Đảo vai trò**: với mỗi cột, đếm theo **cặp hàng** cùng có `1`; cuối cùng cộng `C(v, 2)`.

### 7.1. Cách 2 — Từng cặp hàng, đếm cột chung rồi `C(k, 2)`

Giải thích từng đoạn:

1. Với mỗi cặp hàng `r1 < r2`, đếm `k` = số cột `c` có `grid[r1][c] == 1` và `grid[r2][c] == 1`.
2. Cộng `k·(k-1)//2` vào đáp án (mọi cách chọn hai cột chung).

In [12]:
def count_row_pairs(grid):
    m, n = len(grid), len(grid[0])
    ans = 0
    for r1 in range(m):
        row1 = grid[r1]
        for r2 in range(r1 + 1, m):
            row2 = grid[r2]
            k = 0
            for c in range(n):
                if row1[c] and row2[c]:
                    k += 1
            ans += k * (k - 1) // 2
    return ans

for grid, expected in examples:
    got = count_row_pairs(grid)
    print(f"m={len(grid)}, n={len(grid[0])} -> {got}")
    assert got == expected

m=4, n=5 -> 1
m=3, n=3 -> 9
m=1, n=4 -> 0


In [13]:
class Solution2:
    def countCornerRectangles(self, grid: List[List[int]]) -> int:
        m, n = len(grid), len(grid[0])
        ans = 0
        for r1 in range(m):
            for r2 in range(r1 + 1, m):
                k = sum(1 for c in range(n) if grid[r1][c] and grid[r2][c])
                ans += k * (k - 1) // 2
        return ans

for grid, expected in examples:
    assert Solution2().countCornerRectangles(grid) == expected
print("Solution2 OK")

Solution2 OK


**Độ phức tạp Cách 2:** Time `O(m² n)` (với `200`, cỡ `4·10^6`); Space `O(1)`.

### 7.2. Cách 3 — Bitmask + popcount

Ý tưởng giống Cách 2 nhưng nén mỗi hàng thành một số nguyên: số cột chung của hai hàng là `(mask[r1] & mask[r2]).bit_count()`. Phép `&` trên số nguyên 200 bit chỉ tốn vài chục thao tác máy, nhanh hơn vòng lặp `n` lần trong Python.

Giải thích từng đoạn:

1. `masks = [row_mask(row) for row in grid]` (bit `j` bằng `1` nếu `grid[r][j] == 1`).
2. Với mỗi cặp `(r1, r2)`: `k = (masks[r1] & masks[r2]).bit_count()`; cộng `k*(k-1)//2`.

In [14]:
def count_bitmask(grid):
    masks = [row_mask(row) for row in grid]
    m = len(masks)
    ans = 0
    for r1 in range(m):
        for r2 in range(r1 + 1, m):
            k = (masks[r1] & masks[r2]).bit_count()
            ans += k * (k - 1) // 2
    return ans

for grid, expected in examples:
    got = count_bitmask(grid)
    print(f"m={len(grid)}, n={len(grid[0])} -> {got}")
    assert got == expected

m=4, n=5 -> 1
m=3, n=3 -> 9
m=1, n=4 -> 0


In [15]:
class Solution3:
    def countCornerRectangles(self, grid: List[List[int]]) -> int:
        masks = []
        for row in grid:
            mask = 0
            for j, v in enumerate(row):
                if v:
                    mask |= 1 << j
            masks.append(mask)
        m = len(masks)
        ans = 0
        for r1 in range(m):
            for r2 in range(r1 + 1, m):
                k = (masks[r1] & masks[r2]).bit_count()
                ans += k * (k - 1) // 2
        return ans

for grid, expected in examples:
    assert Solution3().countCornerRectangles(grid) == expected
print("Solution3 OK")

Solution3 OK


**Độ phức tạp Cách 3:** Time `O(n·m)` để dựng mask + `O(m² · n/w)` cho các phép `&` và popcount (`w` là độ rộng từ máy, 64); Space `O(m · n/w)`.

### 7.3. Cách 4 — Bảng băm đếm cặp cột (lời giải nộp)

Giải thích từng đoạn:

1. `cnt` là bảng băm: `cnt[(i, j)]` = số hàng **đã xử lý** có `1` ở cả cột `i` và cột `j` (`i < j`).
2. Với mỗi hàng, với mỗi cặp cột `(i, j)`, `i < j` mà hàng này có `1` ở cả hai: số hình chữ nhật mới (hàng này là cạnh dưới) là `cnt[(i, j)]` (mỗi hàng trước đó có cùng cặp cột cho đúng một hình). Cộng vào `ans`.
3. **Sau đó** mới `cnt[(i, j)] += 1` (nếu tăng trước thì hàng hiện tại bị ghép với chính nó).

Để nhanh, với mỗi hàng ta thu trước danh sách chỉ số cột có `1` (`ones`) rồi xét mọi cặp trong `ones`: tổng công việc là `Σ C(k_r, 2)` theo số ô `1` mỗi hàng.

In [16]:
def count_hash(grid):
    ans = 0
    cnt = Counter()
    for row in grid:
        ones = [c for c, v in enumerate(row) if v]
        for a in range(len(ones)):
            for b in range(a + 1, len(ones)):
                pair = (ones[a], ones[b])
                ans += cnt[pair]          # trước: cộng số hàng trước đó có cùng cặp cột
                cnt[pair] += 1            # sau: ghi nhận hàng hiện tại
    return ans

for grid, expected in examples:
    got = count_hash(grid)
    print(f"m={len(grid)}, n={len(grid[0])} -> {got}")
    assert got == expected

m=4, n=5 -> 1
m=3, n=3 -> 9
m=1, n=4 -> 0


In [17]:
class Solution:
    def countCornerRectangles(self, grid: List[List[int]]) -> int:
        ans = 0
        cnt = Counter()
        n = len(grid[0])
        for row in grid:
            for i, c1 in enumerate(row):
                if c1:
                    for j in range(i + 1, n):
                        if row[j]:
                            ans += cnt[(i, j)]
                            cnt[(i, j)] += 1
        return ans

for grid, expected in examples:
    assert Solution().countCornerRectangles(grid) == expected
print("Solution (cách 4) OK")

Solution (cách 4) OK


**Độ phức tạp Cách 4:** Time `O(Σ C(k_r, 2))` với `k_r` là số ô `1` ở hàng `r`; trong xấu nhất `O(m · n²)`. Do tổng số ô `1` không quá `6000`, tổng `Σ C(k_r, 2)` bị chặn bởi cỡ `6000 · 200 / 2 = 6·10^5`. Space `O(n²)` cho bảng băm (thực tế tối đa số cặp cột xuất hiện).

### 7.4. Cách 5 — Đảo vai trò: đếm theo cặp hàng của từng cột

Đối xứng với Cách 4: **chuyển vị** góc nhìn. Với mỗi **cột** `c`, lấy danh sách các hàng có `1`, rồi với mỗi cặp hàng `(r1, r2)` trong đó tăng `pairs[(r1, r2)]` thêm `1`. Sau khi duyệt hết cột, `pairs[(r1, r2)]` chính là số cột chung `k` của hai hàng đó, và đáp án là `Σ C(k, 2)`.

Cách này làm rõ rằng "đếm cặp rồi áp công thức tổ hợp" là **cùng một đáp án** với việc ghép dần của Cách 4, chỉ khác thứ tự tính. Nó hữu ích khi số hàng ít hơn số cột nhiều (hoặc ngược lại): bạn chọn hướng cho cặp có ít khả năng hơn.

In [18]:
def count_by_columns(grid):
    m, n = len(grid), len(grid[0])
    pairs = Counter()
    for c in range(n):
        rows = [r for r in range(m) if grid[r][c]]
        for a in range(len(rows)):
            for b in range(a + 1, len(rows)):
                pairs[(rows[a], rows[b])] += 1
    return sum(k * (k - 1) // 2 for k in pairs.values())

for grid, expected in examples:
    got = count_by_columns(grid)
    print(f"m={len(grid)}, n={len(grid[0])} -> {got}")
    assert got == expected

m=4, n=5 -> 1
m=3, n=3 -> 9
m=1, n=4 -> 0


In [19]:
class Solution5:
    def countCornerRectangles(self, grid: List[List[int]]) -> int:
        m, n = len(grid), len(grid[0])
        pairs = Counter()
        for c in range(n):
            rows = [r for r in range(m) if grid[r][c]]
            for a in range(len(rows)):
                for b in range(a + 1, len(rows)):
                    pairs[(rows[a], rows[b])] += 1
        return sum(k * (k - 1) // 2 for k in pairs.values())

for grid, expected in examples:
    assert Solution5().countCornerRectangles(grid) == expected
print("Solution5 OK")

Solution5 OK


**Độ phức tạp Cách 5:** Time `O(Σ C(k_c, 2))` với `k_c` là số ô `1` ở cột `c` (tối đa `O(n · m²)`); Space `O(m²)`.

Bảng so sánh năm cách ở mục 16.

## 8. TRACE từng bước

Trace Cách 4 (bảng băm) trên ví dụ 1 (`h0 = {0,3}`, `h1 = {2,4}`, `h2 = {3}`, `h3 = {0,2,4}`):

| hàng | các cột có `1` | cặp cột xét | `cnt` trước | `ans` thêm | `cnt` sau |
|---|---|---|---|---|---|
| h0 | 0, 3 | (0,3) | 0 | 0 | `(0,3):1` |
| h1 | 2, 4 | (2,4) | 0 | 0 | `(0,3):1, (2,4):1` |
| h2 | 3 | (không có cặp) | | 0 | không đổi |
| h3 | 0, 2, 4 | (0,2) | 0 | 0 | `(0,2):1` |
| | | (0,4) | 0 | 0 | `(0,4):1` |
| | | (2,4) | 1 | **1** | `(2,4):2` |

Tổng `ans = 1`. Hàm dưới in đúng từng bước.

In [20]:
def trace_hash(grid):
    ans = 0
    cnt = Counter()
    for r, row in enumerate(grid):
        ones = [c for c, v in enumerate(row) if v]
        print(f"hàng {r}: cột có 1 = {ones}")
        if len(ones) < 2:
            print("   (ít hơn 2 ô 1 -> không có cặp cột)")
        for a in range(len(ones)):
            for b in range(a + 1, len(ones)):
                pair = (ones[a], ones[b])
                add = cnt[pair]
                ans += add
                cnt[pair] += 1
                print(f"   cặp {pair}: đã có {add} hàng trước -> ans += {add} (ans = {ans}); cnt[{pair}] = {cnt[pair]}")
    print("kết quả:", ans)
    return ans

assert trace_hash(examples[0][0]) == 1

hàng 0: cột có 1 = [0, 3]
   cặp (0, 3): đã có 0 hàng trước -> ans += 0 (ans = 0); cnt[(0, 3)] = 1
hàng 1: cột có 1 = [2, 4]
   cặp (2, 4): đã có 0 hàng trước -> ans += 0 (ans = 0); cnt[(2, 4)] = 1
hàng 2: cột có 1 = [3]
   (ít hơn 2 ô 1 -> không có cặp cột)
hàng 3: cột có 1 = [0, 2, 4]
   cặp (0, 2): đã có 0 hàng trước -> ans += 0 (ans = 0); cnt[(0, 2)] = 1
   cặp (0, 4): đã có 0 hàng trước -> ans += 0 (ans = 0); cnt[(0, 4)] = 1
   cặp (2, 4): đã có 1 hàng trước -> ans += 1 (ans = 1); cnt[(2, 4)] = 2
kết quả: 1


Trace ví dụ 2 (lưới `3x3` toàn `1`): hàng đầu không cộng gì (chưa có hàng trước); hàng thứ hai mỗi cặp cột cộng `1` (ba cặp: `+3`); hàng thứ ba mỗi cặp cột cộng `2` (`+6`); tổng `9`.

In [21]:
assert trace_hash(examples[1][0]) == 9
print()
assert trace_hash(examples[2][0]) == 0

hàng 0: cột có 1 = [0, 1, 2]
   cặp (0, 1): đã có 0 hàng trước -> ans += 0 (ans = 0); cnt[(0, 1)] = 1
   cặp (0, 2): đã có 0 hàng trước -> ans += 0 (ans = 0); cnt[(0, 2)] = 1
   cặp (1, 2): đã có 0 hàng trước -> ans += 0 (ans = 0); cnt[(1, 2)] = 1
hàng 1: cột có 1 = [0, 1, 2]
   cặp (0, 1): đã có 1 hàng trước -> ans += 1 (ans = 1); cnt[(0, 1)] = 2
   cặp (0, 2): đã có 1 hàng trước -> ans += 1 (ans = 2); cnt[(0, 2)] = 2
   cặp (1, 2): đã có 1 hàng trước -> ans += 1 (ans = 3); cnt[(1, 2)] = 2
hàng 2: cột có 1 = [0, 1, 2]
   cặp (0, 1): đã có 2 hàng trước -> ans += 2 (ans = 5); cnt[(0, 1)] = 3
   cặp (0, 2): đã có 2 hàng trước -> ans += 2 (ans = 7); cnt[(0, 2)] = 3
   cặp (1, 2): đã có 2 hàng trước -> ans += 2 (ans = 9); cnt[(1, 2)] = 3
kết quả: 9

hàng 0: cột có 1 = [0, 1, 2, 3]
   cặp (0, 1): đã có 0 hàng trước -> ans += 0 (ans = 0); cnt[(0, 1)] = 1
   cặp (0, 2): đã có 0 hàng trước -> ans += 0 (ans = 0); cnt[(0, 2)] = 1
   cặp (0, 3): đã có 0 hàng trước -> ans += 0 (ans = 0); cnt[(0, 3

Trace Cách 3 (bitmask) trên ví dụ 1 để thấy phép `&` tìm cột chung: mỗi hàng là một số nhị phân (in bit từ cột `4` ở trái đến cột `0` ở phải).

In [22]:
def trace_bitmask(grid):
    n = len(grid[0])
    masks = [row_mask(row) for row in grid]
    for r, mk in enumerate(masks):
        print(f"hàng {r}: mask = {mk:0{n}b}  (cột 0 ở bên phải)")
    ans = 0
    for r1 in range(len(masks)):
        for r2 in range(r1 + 1, len(masks)):
            both = masks[r1] & masks[r2]
            k = both.bit_count()
            ans += k * (k - 1) // 2
            print(f"cặp ({r1},{r2}): {masks[r1]:0{n}b} & {masks[r2]:0{n}b} = {both:0{n}b} -> k = {k} -> C(k,2) = {k * (k - 1) // 2}")
    print("kết quả:", ans)
    return ans

assert trace_bitmask(examples[0][0]) == 1

hàng 0: mask = 01001  (cột 0 ở bên phải)
hàng 1: mask = 10100  (cột 0 ở bên phải)
hàng 2: mask = 01000  (cột 0 ở bên phải)
hàng 3: mask = 10101  (cột 0 ở bên phải)
cặp (0,1): 01001 & 10100 = 00000 -> k = 0 -> C(k,2) = 0
cặp (0,2): 01001 & 01000 = 01000 -> k = 1 -> C(k,2) = 0
cặp (0,3): 01001 & 10101 = 00001 -> k = 1 -> C(k,2) = 0
cặp (1,2): 10100 & 01000 = 00000 -> k = 0 -> C(k,2) = 0
cặp (1,3): 10100 & 10101 = 10100 -> k = 2 -> C(k,2) = 1
cặp (2,3): 01000 & 10101 = 00000 -> k = 0 -> C(k,2) = 0
kết quả: 1


## 9. Code nộp LeetCode

Ta chọn **Cách 4 (bảng băm đếm cặp cột)**: đúng với ràng buộc (≤ 6000 ô `1`), ngắn gọn và là lời giải chuẩn.

In [23]:
class Solution:
    def countCornerRectangles(self, grid: List[List[int]]) -> int:
        ans = 0
        cnt = Counter()
        n = len(grid[0])
        for row in grid:
            for i, c1 in enumerate(row):
                if c1:
                    for j in range(i + 1, n):
                        if row[j]:
                            ans += cnt[(i, j)]
                            cnt[(i, j)] += 1
        return ans


sol = Solution()
for grid, expected in examples:
    got = sol.countCornerRectangles(grid)
    print(f"m={len(grid)}, n={len(grid[0])} -> {got}")
    assert got == expected

m=4, n=5 -> 1
m=3, n=3 -> 9
m=1, n=4 -> 0


## 10. Vì sao đúng

**Khẳng định 1 (tương ứng một-một).** Một hình chữ nhật góc là một bộ `(r1, r2, c1, c2)` với `r1 < r2`, `c1 < c2` và bốn góc đều bằng `1` (bốn ô phân biệt chính là điều kiện `r1 != r2`, `c1 != c2`; có thể chọn sắp xếp tăng để mỗi hình xuất hiện đúng một lần). Gọi `S(r)` là tập các cột có `1` ở hàng `r`. Điều kiện bốn góc bằng `1` tương đương `c1, c2 ∈ S(r1) ∩ S(r2)`.

**Khẳng định 2 (công thức theo cặp hàng).** Với mỗi cặp hàng `r1 < r2`, số hình chữ nhật nhận hai hàng này làm cạnh trên/dưới bằng số cặp `{c1 < c2}` trong `S(r1) ∩ S(r2)`, tức `C(|S(r1) ∩ S(r2)|, 2)`. Cộng theo mọi cặp hàng ta được đáp án (Cách 2 và 3). Cách 3 chỉ tính `|S(r1) ∩ S(r2)|` bằng `&` và popcount.

**Khẳng định 3 (Cách 4 đúng).** Xét hàng `r` và cặp cột `(i, j)`, `i < j` thuộc `S(r)`. Trước khi xử lý `r`, `cnt[(i, j)]` bằng số hàng `r' < r` mà `(i, j)` cũng thuộc `S(r')`, tức số hàng `r'` tạo được hình chữ nhật `(r', r, i, j)`. Do đó `ans += cnt[(i, j)]` thêm đúng số hình mà hàng `r` là **cạnh dưới** và cặp cột là `(i, j)`. Mỗi hình `(r1, r2, c1, c2)` với `r1 < r2` được đếm đúng một lần: khi xử lý hàng `r2` tại cặp `(c1, c2)`. Phải cộng **trước** khi tăng `cnt`, nếu không hàng `r` bị tính ghép với chính nó.

**Khẳng định 4 (Cách 5 đúng).** Sau khi duyệt mọi cột, `pairs[(r1, r2)]` bằng số cột `c` có `1` ở cả `r1` và `r2`, tức `|S(r1) ∩ S(r2)|`. Áp dụng Khẳng định 2.

**Cận thời gian của Cách 4.** Với hàng có `k_r` ô `1` ta xét `C(k_r, 2)` cặp. Vì `Σ k_r <= 6000` và `k_r <= n <= 200`, ta có `Σ C(k_r, 2) <= Σ k_r · n / 2 <= 6000 · 200 / 2 = 6 · 10^5`. Đó là lý do cách này rất nhanh dù lưới `200 x 200`.

## 11. Time / Space Complexity

- **Cách 1 (bốn vòng lặp):** Time `O(m² n²)`; Space `O(1)`.
- **Cách 2 (từng cặp hàng):** Time `O(m² n)`; Space `O(1)`.
- **Cách 3 (bitmask):** Time `O(m² · n / 64)` + dựng mask `O(mn)`; Space `O(m · n / 64)`.
- **Cách 4 (bảng băm, nộp):** Time `O(Σ C(k_r, 2)) ⊆ O(m n²)`, tối đa cỡ `6·10^5` với ràng buộc; Space `O(n²)`.
- **Cách 5 (theo cột):** Time `O(Σ C(k_c, 2)) ⊆ O(n m²)`; Space `O(m²)`.

Hãy đo trên các lưới cỡ tối đa (200x200): lưới ngẫu nhiên có ~6000 ô `1`, lưới "dày" `30 x 200` toàn `1` (6000 ô), và chuyển vị của nó (`200 x 30`).

In [24]:
def dense_grid(m, n):
    return [[1] * n for _ in range(m)]

rng = random.Random(3)
rand_grid = [[0] * 200 for _ in range(200)]
cells = rng.sample(range(200 * 200), 6000)
for x in cells:
    rand_grid[x // 200][x % 200] = 1

grids = {
    "ngẫu nhiên 200x200, 6000 ô 1": rand_grid,
    "dày 30x200 toàn 1": dense_grid(30, 200),
    "dày 200x30 toàn 1": dense_grid(200, 30),
}
methods = [
    ("row-pairs", count_row_pairs),
    ("bitmask", count_bitmask),
    ("hash", count_hash),
    ("by-columns", count_by_columns),
]
for name, g in grids.items():
    results = []
    for mname, fn in methods:
        res, dt = timeit_once(fn, g)
        results.append((mname, res, dt))
    assert len({r[1] for r in results}) == 1
    print(f"{name:30s} đáp án = {results[0][1]:>10d} | " + " | ".join(f"{m}: {dt * 1000:7.1f} ms" for m, _, dt in results))
assert count_hash(dense_grid(30, 200)) == comb(30, 2) * comb(200, 2)

ngẫu nhiên 200x200, 6000 ô 1   đáp án =     198480 | row-pairs:    78.4 ms | bitmask:     3.4 ms | hash:    26.9 ms | by-columns:    21.4 ms

dày 30x200 toàn 1              đáp án =    8656500 | row-pairs:     2.8 ms | bitmask:     0.5 ms | hash:   132.5 ms | by-columns:    14.4 ms


dày 200x30 toàn 1              đáp án =    8656500 | row-pairs:    22.1 ms | bitmask:     2.3 ms | hash:    18.6 ms | by-columns:   100.5 ms


Cả bốn cách tuyến tính-theo-cặp đều chạy dưới vài giây ở cỡ tối đa; `bitmask` và `hash` nhanh nhất. Bốn vòng lặp ở `200 x 200` cần cỡ `4·10^8` bước nên không thể chạy trong notebook.

## 12. Bộ test

Bốn lớp test: ví dụ đề, edge case, phản ví dụ cho các cách "gần đúng", và đối chiếu có hệ thống (vét cạn mọi lưới nhỏ) cùng ngẫu nhiên có seed với oracle `count_brute` (bốn vòng lặp làm đúng định nghĩa).

In [25]:
all_methods = {
    "row-pairs": count_row_pairs,
    "bitmask": count_bitmask,
    "hash": count_hash,
    "by-columns": count_by_columns,
    "Sol2": lambda g: Solution2().countCornerRectangles(g),
    "Sol3": lambda g: Solution3().countCornerRectangles(g),
    "Sol5": lambda g: Solution5().countCornerRectangles(g),
    "submit": lambda g: Solution().countCornerRectangles(g),
}

def run_all(grid, expected=None):
    exp = count_brute(grid) if expected is None else expected
    for name, fn in all_methods.items():
        got = fn(grid)
        assert got == exp, (name, grid, got, exp)
    return exp

for grid, expected in examples:
    assert run_all(grid, expected) == expected
print("ví dụ đề: OK")

ví dụ đề: OK


In [26]:
edge = [
    ([[1]], 0),                                   # một ô
    ([[0]], 0),                                   # (ngoài ràng buộc, vẫn an toàn)
    ([[1, 1]], 0),                                # một hàng
    ([[1], [1]], 0),                              # một cột
    ([[1, 1], [1, 1]], 1),                        # hình chữ nhật nhỏ nhất
    ([[1, 1], [1, 0]], 0),                        # thiếu một góc
    ([[1, 0, 1], [0, 1, 0], [1, 0, 1]], 1),       # bốn góc ở bốn đỉnh lưới, tâm khác
    ([[1, 1, 1], [1, 1, 1]], 3),                  # k = 3 cột chung -> C(3,2) = 3
    ([[1, 1, 1], [1, 1, 1], [1, 1, 1]], 9),
    ([[1, 1, 1, 1], [1, 1, 1, 1]], 6),            # C(4,2) = 6
    ([[1, 1], [1, 1], [1, 1]], 3),                # 3 hàng, 2 cột: C(3,2) = 3
    ([[1, 0, 1], [1, 0, 1], [1, 0, 1]], 3),       # cột giữa toàn 0
    ([[1, 0], [0, 1]], 0),                        # đường chéo
]
for grid, expected in edge:
    got = run_all(grid)
    print(grid, "->", got)
    assert got == expected
print("edge case: OK")

[[1]] -> 0
[[0]] -> 0
[[1, 1]] -> 0
[[1], [1]] -> 0
[[1, 1], [1, 1]] -> 1
[[1, 1], [1, 0]] -> 0
[[1, 0, 1], [0, 1, 0], [1, 0, 1]] -> 1
[[1, 1, 1], [1, 1, 1]] -> 3
[[1, 1, 1], [1, 1, 1], [1, 1, 1]] -> 9
[[1, 1, 1, 1], [1, 1, 1, 1]] -> 6
[[1, 1], [1, 1], [1, 1]] -> 3
[[1, 0, 1], [1, 0, 1], [1, 0, 1]] -> 3
[[1, 0], [0, 1]] -> 0
edge case: OK


Phản ví dụ cho các cách "gần đúng" người mới hay nghĩ ra:

- *Cộng `k` (số cột chung) thay vì `C(k, 2)`*: với `[[1,1,1],[1,1,1]]` cả hai cách đều ra `3` (trùng hợp vì `C(3,2) = 3`), nhưng với `[[1,1,1,1],[1,1,1,1]]` đúng là `6` còn cộng `k` chỉ cho `4`.
- *Đếm cặp hàng hai lần* (cả `(r1, r2)` và `(r2, r1)`): kết quả gấp đôi.
- *Chỉ kiểm tra "mọi hàng có ít nhất hai ô 1"*: không phải điều kiện đủ (ví dụ đường chéo).

In [27]:
def wrong_add_k(grid):
    m, n = len(grid), len(grid[0])
    return sum(sum(1 for c in range(n) if grid[a][c] and grid[b][c]) for a in range(m) for b in range(a + 1, m))

def wrong_ordered_pairs(grid):
    m, n = len(grid), len(grid[0])
    ans = 0
    for a in range(m):
        for b in range(m):
            if a != b:
                k = sum(1 for c in range(n) if grid[a][c] and grid[b][c])
                ans += k * (k - 1) // 2
    return ans

g = [[1, 1, 1, 1], [1, 1, 1, 1]]
print("cộng k           :", wrong_add_k(g), "| đúng:", count_brute(g))
print("cặp có thứ tự    :", wrong_ordered_pairs(g), "| đúng:", count_brute(g))
assert wrong_add_k(g) != count_brute(g) and wrong_ordered_pairs(g) != count_brute(g)

cộng k           : 4 | đúng: 6
cặp có thứ tự    : 12 | đúng: 6


In [28]:
# Đối chiếu có hệ thống: MỌI lưới nhị phân 3x3 (512), 3x4 và 4x3 (4096 mỗi loại), 4x4 (65536)
def all_grids(m, n):
    for bits in range(1 << (m * n)):
        yield [[(bits >> (r * n + c)) & 1 for c in range(n)] for r in range(m)]

total = 0
for (m, n) in ((3, 3), (3, 4), (4, 3), (4, 4)):
    for g in all_grids(m, n):
        run_all(g)
        total += 1
print("vét cạn", total, "lưới nhỏ: mọi cách khớp oracle")

vét cạn 74240 lưới nhỏ: mọi cách khớp oracle


In [29]:
# Ngẫu nhiên có seed, nhiều kích thước và mật độ
rng = random.Random(2028)
for t in range(500):
    m = rng.randint(1, 12)
    n = rng.randint(1, 12)
    p = rng.choice([0.1, 0.3, 0.5, 0.8, 1.0])
    g = random_grid(rng, m, n, p)
    run_all(g)
print("500 lưới ngẫu nhiên: OK")

# Lưới lớn: đáp án biết trước / đối chiếu giữa các cách nhanh
dense = dense_grid(30, 200)
for name in ("bitmask", "hash", "by-columns", "Sol3", "Sol5", "submit"):
    assert all_methods[name](dense) == comb(30, 2) * comb(200, 2)
rng = random.Random(9)
for p in (0.05, 0.1, 0.15):
    g = random_grid(rng, 200, 200, p)
    while sum(map(sum, g)) > 6000:
        g[rng.randrange(200)][rng.randrange(200)] = 0
    base = count_bitmask(g)
    for name in ("hash", "by-columns", "Sol5", "submit", "row-pairs"):
        assert all_methods[name](g) == base, name
print("test lưới 200x200: OK")

500 lưới ngẫu nhiên: OK


test lưới 200x200: OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Cộng `k` thay vì `C(k, 2)`

Code sai: `ans += k` (số cột chung). Mỗi cặp cột chung cho một hình, không phải mỗi cột. Với `k = 4` có `6` hình chứ không phải `4`.

In [30]:
g = [[1, 1, 1, 1], [1, 1, 1, 1]]
print("sai (cộng k)   :", wrong_add_k(g))
print("đúng (C(k,2))  :", Solution().countCornerRectangles(g))
assert Solution().countCornerRectangles(g) == comb(4, 2) == 6

sai (cộng k)   : 4
đúng (C(k,2))  : 6


### Lỗi 2 — Cộng sau khi tăng bảng băm (hoặc tăng trước khi cộng)

Trong Cách 4 thứ tự là: **cộng `cnt[pair]` trước, rồi mới `cnt[pair] += 1`**. Nếu đảo thứ tự, hàng hiện tại bị ghép với chính nó, đáp án thừa. Với lưới `[[1,1],[1,1]]` đúng là `1` nhưng bản sai cho `3`.

In [31]:
def buggy_increment_first(grid):
    ans = 0
    cnt = Counter()
    n = len(grid[0])
    for row in grid:
        for i in range(n):
            if row[i]:
                for j in range(i + 1, n):
                    if row[j]:
                        cnt[(i, j)] += 1              # BUG: tăng trước
                        ans += cnt[(i, j)]
    return ans

for g in ([[1, 1], [1, 1]], [[1, 1, 1], [1, 1, 1], [1, 1, 1]], [[1, 1, 1, 1]]):
    print(g, "| sai:", buggy_increment_first(g), "| đúng:", count_brute(g))
assert buggy_increment_first([[1, 1, 1, 1]]) != 0

[[1, 1], [1, 1]] | sai: 3 | đúng: 1
[[1, 1, 1], [1, 1, 1], [1, 1, 1]] | sai: 18 | đúng: 9
[[1, 1, 1, 1]] | sai: 6 | đúng: 0


### Lỗi 3 — Quên rằng bốn góc phải phân biệt (một hàng cũng "đếm")

Với một hàng toàn `1`, ví dụ `[[1,1,1,1]]`, bốn ô nằm trên cùng một hàng không tạo hình chữ nhật: cần **hai hàng khác nhau**. Nếu bạn lập công thức `C(tổng số cột có 1, 2)` mà không phân theo hàng, kết quả sai (cho ví dụ 3 ra `6` thay vì `0`).

In [32]:
def wrong_ignore_rows(grid):
    n = len(grid[0])
    cols_with_one = sum(1 for c in range(n) if any(row[c] for row in grid))
    return comb(cols_with_one, 2)

for g in ([[1, 1, 1, 1]], [[1, 0], [0, 1]], [[1, 1], [1, 1]]):
    print(g, "| sai:", wrong_ignore_rows(g), "| đúng:", count_brute(g))
assert wrong_ignore_rows([[1, 1, 1, 1]]) == 6 != count_brute([[1, 1, 1, 1]])

[[1, 1, 1, 1]] | sai: 6 | đúng: 0
[[1, 0], [0, 1]] | sai: 1 | đúng: 0
[[1, 1], [1, 1]] | sai: 1 | đúng: 1


### Lỗi 4 — Quên điều kiện `i < j` (đếm mỗi cặp cột hai lần)

Nếu xét cả `(i, j)` và `(j, i)` (hoặc cả `i == j`), mỗi hình bị đếm hai lần hoặc có hình "suy biến" (hai góc trùng nhau). Hãy dùng `j in range(i + 1, n)` để mỗi cặp chỉ xuất hiện một lần.

In [33]:
def buggy_both_orders(grid):
    ans = 0
    cnt = Counter()
    n = len(grid[0])
    for row in grid:
        for i in range(n):
            for j in range(n):                 # BUG: bỏ i < j
                if i != j and row[i] and row[j]:
                    ans += cnt[(i, j)]
                    cnt[(i, j)] += 1
    return ans

for g in ([[1, 1], [1, 1]], [[1, 1, 1], [1, 1, 1], [1, 1, 1]]):
    print(g, "| sai:", buggy_both_orders(g), "| đúng:", count_brute(g))
assert buggy_both_orders([[1, 1], [1, 1]]) == 2 * count_brute([[1, 1], [1, 1]])

[[1, 1], [1, 1]] | sai: 2 | đúng: 1
[[1, 1, 1], [1, 1, 1], [1, 1, 1]] | sai: 18 | đúng: 9


### Lỗi 5 — Dùng cách `O(m² n²)` cho `200 x 200`

Bốn vòng lặp đúng nhưng quá chậm: Ở `40 x 40` thời gian đã tăng rõ rệt (mục 5); `200 x 200` lớn hơn `5^4 = 625` lần. Hãy luôn ước lượng số phép tính: `C(200,2)² ≈ 4·10^8`. Dưới đây ta đo `60 x 60` rồi ngoại suy.

In [34]:
rng = random.Random(4)
g60 = random_grid(rng, 60, 60, 0.3)
res, dt60 = timeit_once(count_brute, g60)
print(f"60x60: bốn vòng lặp mất {dt60 * 1000:.0f} ms; ngoại suy 200x200 ~ {dt60 * (200 / 60) ** 4:.0f} giây")
assert res == count_hash(g60) == count_bitmask(g60)
g200, = [rand_grid]
res, dt = timeit_once(count_hash, g200)
print(f"200x200 (6000 ô 1): bảng băm mất {dt * 1000:.0f} ms")

60x60: bốn vòng lặp mất 117 ms; ngoại suy 200x200 ~ 14 giây
200x200 (6000 ô 1): bảng băm mất 26 ms


## 14. Mẹo & nhận dạng pattern

- **Nhận dạng:** "đếm hình chữ nhật/bộ bốn góc trong lưới" ⇒ nghĩ ngay tới **cặp** (cặp hàng hoặc cặp cột) và công thức **`C(k, 2)`**, thay vì liệt kê bộ bốn.
- Công thức "đếm rồi tổ hợp": mỗi cặp hàng có `k` cột chung ⇒ `C(k, 2)` hình.
- Ràng buộc có chặn **số ô `1`** (≤ 6000) là gợi ý dùng thuật toán **phụ thuộc số ô 1** (cặp cột có `1` cả hai) thay vì quét toàn lưới.
- Khi cộng dồn theo thứ tự: **cộng trước, tăng sau** để không tự ghép.
- Bitmask + `int.bit_count()` là cách nhanh để đếm giao của hai tập cột trong Python.
- Chọn hướng (cặp hàng hay cặp cột) theo chiều **nhỏ hơn** nếu hình dạng lưới lệch (ví dụ `200 x 30`).
- Các bài cùng họ: **1139. Largest 1-Bordered Square**, **221. Maximal Square**, **85. Maximal Rectangle**, **1727. Largest Submatrix With Rearrangements**, **1074. Number of Submatrices That Sum to Target**.

## 15. Biến thể

### 15.1. Chuyển vị lưới cho cùng đáp án

Hình chữ nhật nhìn từ hàng hay từ cột đều như nhau: đáp án của lưới và của lưới chuyển vị bằng nhau. Ta có thể chọn hướng rẻ hơn.

In [35]:
def transpose(grid):
    return [list(col) for col in zip(*grid)]

rng = random.Random(15)
for _ in range(200):
    g = random_grid(rng, rng.randint(1, 10), rng.randint(1, 10), rng.choice([0.3, 0.6, 1.0]))
    assert count_hash(g) == count_hash(transpose(g))
print("đáp án của lưới = đáp án của lưới chuyển vị: OK")

def count_cheapest_direction(grid):
    t = transpose(grid)
    cost_rows = sum(comb(sum(row), 2) for row in grid)
    cost_cols = sum(comb(sum(row), 2) for row in t)
    return count_hash(grid if cost_rows <= cost_cols else t)

assert count_cheapest_direction(dense_grid(200, 30)) == comb(200, 2) * comb(30, 2)
print("chọn hướng rẻ hơn: OK")

đáp án của lưới = đáp án của lưới chuyển vị: OK


chọn hướng rẻ hơn: OK


### 15.2. Liệt kê chính các hình chữ nhật (không chỉ đếm)

Từ cặp hàng và các cột chung, ta sinh từng bộ `(r1, r2, c1, c2)`.

In [36]:
def list_rectangles(grid):
    out = []
    for r1, r2 in combinations(range(len(grid)), 2):
        cols = common_columns(grid, r1, r2)
        for c1, c2 in combinations(cols, 2):
            out.append((r1, r2, c1, c2))
    return out

rects = list_rectangles(examples[0][0])
print(rects)
assert rects == [(1, 3, 2, 4)]
rects = list_rectangles(examples[1][0])
print(len(rects), "hình, đầu tiên:", rects[:3])
assert len(rects) == 9
for g in (random_grid(random.Random(3), 8, 8, 0.5),):
    assert len(list_rectangles(g)) == count_hash(g)

[(1, 3, 2, 4)]
9 hình, đầu tiên: [(0, 1, 0, 1), (0, 1, 0, 2), (0, 1, 1, 2)]


### 15.3. Đếm **hình vuông góc** (corner squares)

Nếu yêu cầu thêm "chiều cao = chiều rộng" (`r2 - r1 == c2 - c1`), ta lọc các bộ ở trên. Với lưới toàn `1` cỡ `m x n` số hình vuông góc là `Σ_{d=1}^{min(m,n)-1} (m-d)(n-d)`.

In [37]:
def count_corner_squares(grid):
    return sum(1 for (r1, r2, c1, c2) in list_rectangles(grid) if r2 - r1 == c2 - c1)

def formula_all_ones(m, n):
    return sum((m - d) * (n - d) for d in range(1, min(m, n)))

for m, n in ((2, 2), (3, 3), (4, 6), (5, 5)):
    got = count_corner_squares(dense_grid(m, n))
    print(f"toàn 1 {m}x{n}: {got} hình vuông góc, công thức {formula_all_ones(m, n)}")
    assert got == formula_all_ones(m, n)

toàn 1 2x2: 1 hình vuông góc, công thức 1
toàn 1 3x3: 5 hình vuông góc, công thức 5
toàn 1 4x6: 26 hình vuông góc, công thức 26
toàn 1 5x5: 30 hình vuông góc, công thức 30


### 15.4. Bốn góc **cùng giá trị** bất kỳ (không chỉ `1`)

Tổng quát: đếm hình chữ nhật có bốn góc cùng bằng một giá trị `v`. Chỉ cần "lọc" theo giá trị: với `v = 1` ta được bài này; với `v = 0` là bài trên **lưới đảo bit**.

In [38]:
def count_corners_with_value(grid, v):
    return count_hash([[1 if x == v else 0 for x in row] for row in grid])

g = [[1, 0, 0], [0, 0, 1], [1, 0, 0], [0, 0, 1]]
print("góc toàn 1:", count_corners_with_value(g, 1), "| góc toàn 0:", count_corners_with_value(g, 0))
rng = random.Random(8)
for _ in range(200):
    grid = random_grid(rng, rng.randint(1, 8), rng.randint(1, 8), 0.5)
    for v in (0, 1):
        brute = sum(
            1
            for r1, r2 in combinations(range(len(grid)), 2)
            for c1, c2 in combinations(range(len(grid[0])), 2)
            if grid[r1][c1] == grid[r1][c2] == grid[r2][c1] == grid[r2][c2] == v
        )
        assert count_corners_with_value(grid, v) == brute
print("tổng quát hóa đúng theo vét cạn")

góc toàn 1: 0 | góc toàn 0: 2
tổng quát hóa đúng theo vét cạn


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Bốn vòng lặp | Duyệt mọi bộ 4 góc | `O(m² n²)` | `O(1)` | Chỉ làm oracle trên lưới nhỏ |
| 2. Từng cặp hàng | Đếm cột chung `k`, cộng `C(k,2)` | `O(m² n)` | `O(1)` | Dễ hiểu; đủ cho `m, n <= 200` |
| 3. Bitmask | `&` và `bit_count` thay vòng đếm | `O(m² n / 64)` | `O(m n / 64)` | Nhanh nhất khi lưới dày |
| 4. Bảng băm cặp cột (nộp) | `cnt[(i,j)]`: cộng trước, tăng sau | `O(Σ C(k_r,2))` | `O(n²)` | Khi số ô `1` bị chặn (như đề) |
| 5. Theo cột | Đếm cặp hàng của từng cột, rồi `ΣC(v,2)` | `O(Σ C(k_c,2))` | `O(m²)` | Khi số hàng ít hơn số cột |

Cách 4 và 5 tận dụng độ thưa; cách 3 tận dụng phần cứng. Với ràng buộc của đề, cả ba đều chạy thoải mái.

## 17. Bài tập vận dụng

Làm tuần tự từ dễ đến khó. Các cell có chữ `TODO` **không được gọi** khi chạy notebook; bạn tự điền rồi bỏ dấu `#` ở các dòng `assert` để kiểm tra. Đáp án nằm ở mục 19.

### Bài 1 — Đoán output (dễ)

Không chạy code, hãy đoán output của ba dòng sau, rồi chạy để kiểm tra:

```python
print(bin(0b1011 & 0b0110))
print((0b1011 & 0b0110).bit_count())
print(comb(5, 2))
```

In [39]:
# Bài 1: viết dự đoán của bạn vào biến bên dưới (chưa chạy code mẫu)
my_guess_line1 = None
my_guess_line2 = None
my_guess_line3 = None

### Bài 2 — Tính tay (dễ-vừa)

Với `grid = [[1,1,0,1],[1,1,1,1],[0,1,1,1]]`: viết tập cột có `1` của từng hàng, số cột chung của từng cặp hàng, và đáp án.

In [40]:
# Bài 2: điền kết quả tính tay
hand_common = None      # dict {(r1, r2): số cột chung} cho 3 cặp hàng
hand_answer = None      # đáp án

### Bài 3 — Điền chỗ trống (vừa)

Hoàn thành hàm theo Cách 4 (bảng băm đếm cặp cột).

In [41]:
def exercise_3_hash(grid: List[List[int]]) -> int:
    ans = 0
    cnt = Counter()
    n = len(grid[0])
    # TODO 1: với mỗi hàng row trong grid
    #     TODO 2: với mỗi cặp cột (i, j) mà i < j và row[i] == row[j] == 1
    #         TODO 3: ans += cnt[(i, j)]  (cộng TRƯỚC)
    #         TODO 4: cnt[(i, j)] += 1    (tăng SAU)
    return None


# assert exercise_3_hash([[1, 0, 0, 1, 0], [0, 0, 1, 0, 1], [0, 0, 0, 1, 0], [1, 0, 1, 0, 1]]) == 1
# assert exercise_3_hash([[1, 1, 1], [1, 1, 1], [1, 1, 1]]) == 9
# assert exercise_3_hash([[1, 1, 1, 1]]) == 0

### Bài 4 — Tự viết lại (vừa-khó)

Không nhìn lại Cách 3, hãy tự viết `exercise_4_bitmask(grid)`: nén mỗi hàng thành một số nguyên rồi dùng `&` và `bit_count()` cho từng cặp hàng.

In [42]:
def exercise_4_bitmask(grid: List[List[int]]) -> int:
    # TODO
    pass


# assert exercise_4_bitmask([[1, 0, 0, 1, 0], [0, 0, 1, 0, 1], [0, 0, 0, 1, 0], [1, 0, 1, 0, 1]]) == 1
# assert exercise_4_bitmask([[1, 1, 1], [1, 1, 1], [1, 1, 1]]) == 9
# assert exercise_4_bitmask([[1, 1, 1, 1]]) == 0

### Bài 5 — Mini challenge: diện tích lớn nhất của hình chữ nhật góc (khó)

Viết `exercise_5_max_area(grid)` trả về **diện tích lớn nhất** `(r2 - r1 + 1) * (c2 - c1 + 1)` của một hình chữ nhật góc (bốn góc đều là `1`), hoặc `0` nếu không có hình nào. Gợi ý: với mỗi cặp hàng, hai cột chung **xa nhau nhất** (nhỏ nhất và lớn nhất trong các cột chung) cho diện tích lớn nhất của cặp hàng đó.

```python
exercise_5_max_area([[1,0,0,1,0],[0,0,1,0,1],[0,0,0,1,0],[1,0,1,0,1]]) -> 9   # hàng 1..3, cột 2..4: cao 3, rộng 3
exercise_5_max_area([[1,1,1],[1,1,1],[1,1,1]]) -> 9
exercise_5_max_area([[1,1,1,1]]) -> 0
```

In [43]:
def exercise_5_max_area(grid: List[List[int]]) -> int:
    # TODO
    pass


# assert exercise_5_max_area([[1, 0, 0, 1, 0], [0, 0, 1, 0, 1], [0, 0, 0, 1, 0], [1, 0, 1, 0, 1]]) == 9
# assert exercise_5_max_area([[1, 1, 1], [1, 1, 1], [1, 1, 1]]) == 9
# assert exercise_5_max_area([[1, 1, 1, 1]]) == 0

## 18. Gợi ý 3 mức

**Bài 1**
- Mức 1: `&` giữ các bit mà cả hai cùng `1`; `bit_count` đếm số bit `1`; `comb(5, 2)` là số cách chọn 2 trong 5.
- Mức 2: `1011 & 0110 = 0010`.
- Mức 3: `'0b10'`, `1`, `10`.

**Bài 2**
- Mức 1: tập cột có `1`: `h0 = {0,1,3}`, `h1 = {0,1,2,3}`, `h2 = {1,2,3}`.
- Mức 2: `(h0,h1)`: `{0,1,3}`; `(h0,h2)`: `{1,3}`; `(h1,h2)`: `{1,2,3}`.
- Mức 3: số cột chung `3, 2, 3` nên `C(3,2)+C(2,2)+C(3,2) = 3+1+3 = 7`.

**Bài 3**
- Mức 1: giống mục 7.3.
- Mức 2: `for i, c1 in enumerate(row): if c1: for j in range(i + 1, n): if row[j]: ...`.
- Mức 3: `ans += cnt[(i, j)]` rồi `cnt[(i, j)] += 1`, cuối hàm `return ans`.

**Bài 4**
- Mức 1: `mask |= 1 << j` khi `grid[r][j] == 1`.
- Mức 2: với mỗi `r1 < r2`, `k = (masks[r1] & masks[r2]).bit_count()`.
- Mức 3: cộng `k * (k - 1) // 2` vào đáp án.

**Bài 5**
- Mức 1: nghĩ theo cặp hàng `(r1, r2)` và các cột chung.
- Mức 2: nếu cột chung nhỏ nhất là `lo` và lớn nhất là `hi` (cần ít nhất hai cột chung) thì diện tích `(r2 - r1 + 1) * (hi - lo + 1)`.
- Mức 3: lấy max qua mọi cặp hàng; mặc định `0`. Ví dụ 1 cho `3 * 3 = 9`.

## 19. Đáp án tham khảo

### Đáp án Bài 1

In [44]:
ans1 = [bin(0b1011 & 0b0110), (0b1011 & 0b0110).bit_count(), comb(5, 2)]
print(ans1)
assert ans1 == ["0b10", 1, 10]

['0b10', 1, 10]


### Đáp án Bài 2

In [45]:
ans2_common = {(0, 1): 3, (0, 2): 2, (1, 2): 3}
ans2_answer = 7
g = [[1, 1, 0, 1], [1, 1, 1, 1], [0, 1, 1, 1]]
for (r1, r2), k in ans2_common.items():
    assert len(common_columns(g, r1, r2)) == k
assert sum(comb(k, 2) for k in ans2_common.values()) == ans2_answer == Solution().countCornerRectangles(g) == count_brute(g)
print("Bài 2 OK")

Bài 2 OK


### Đáp án Bài 3

In [46]:
def ans3_hash(grid: List[List[int]]) -> int:
    ans = 0
    cnt = Counter()
    n = len(grid[0])
    for row in grid:
        for i in range(n):
            if row[i]:
                for j in range(i + 1, n):
                    if row[j]:
                        ans += cnt[(i, j)]
                        cnt[(i, j)] += 1
    return ans

assert ans3_hash([[1, 0, 0, 1, 0], [0, 0, 1, 0, 1], [0, 0, 0, 1, 0], [1, 0, 1, 0, 1]]) == 1
assert ans3_hash([[1, 1, 1], [1, 1, 1], [1, 1, 1]]) == 9
assert ans3_hash([[1, 1, 1, 1]]) == 0
rng = random.Random(31)
for _ in range(200):
    g = random_grid(rng, rng.randint(1, 8), rng.randint(1, 8), rng.choice([0.3, 0.7]))
    assert ans3_hash(g) == count_brute(g)
print("Bài 3 OK")

Bài 3 OK


### Đáp án Bài 4

In [47]:
def ans4_bitmask(grid: List[List[int]]) -> int:
    masks = []
    for row in grid:
        mask = 0
        for j, v in enumerate(row):
            if v:
                mask |= 1 << j
        masks.append(mask)
    ans = 0
    for r1 in range(len(masks)):
        for r2 in range(r1 + 1, len(masks)):
            k = (masks[r1] & masks[r2]).bit_count()
            ans += k * (k - 1) // 2
    return ans

assert ans4_bitmask([[1, 0, 0, 1, 0], [0, 0, 1, 0, 1], [0, 0, 0, 1, 0], [1, 0, 1, 0, 1]]) == 1
assert ans4_bitmask([[1, 1, 1], [1, 1, 1], [1, 1, 1]]) == 9
assert ans4_bitmask([[1, 1, 1, 1]]) == 0
rng = random.Random(32)
for _ in range(200):
    g = random_grid(rng, rng.randint(1, 8), rng.randint(1, 8), rng.choice([0.3, 0.7]))
    assert ans4_bitmask(g) == count_brute(g)
print("Bài 4 OK")

Bài 4 OK


### Đáp án Bài 5

In [48]:
def ans5_max_area(grid: List[List[int]]) -> int:
    best = 0
    m = len(grid)
    for r1 in range(m):
        for r2 in range(r1 + 1, m):
            cols = common_columns(grid, r1, r2)
            if len(cols) >= 2:
                best = max(best, (r2 - r1 + 1) * (cols[-1] - cols[0] + 1))
    return best

def max_area_brute(grid):
    best = 0
    for r1, r2, c1, c2 in list_rectangles(grid):
        best = max(best, (r2 - r1 + 1) * (c2 - c1 + 1))
    return best

assert ans5_max_area([[1, 0, 0, 1, 0], [0, 0, 1, 0, 1], [0, 0, 0, 1, 0], [1, 0, 1, 0, 1]]) == 9
assert ans5_max_area([[1, 1, 1], [1, 1, 1], [1, 1, 1]]) == 9
assert ans5_max_area([[1, 1, 1, 1]]) == 0
rng = random.Random(33)
for _ in range(300):
    g = random_grid(rng, rng.randint(1, 8), rng.randint(1, 8), rng.choice([0.3, 0.6, 0.9]))
    assert ans5_max_area(g) == max_area_brute(g)
print("Bài 5 OK")

Bài 5 OK


## 20. Template nhớ nhanh + Checklist trước khi submit

### Template

```python
class Solution:
    def countCornerRectangles(self, grid: List[List[int]]) -> int:
        ans = 0
        cnt = Counter()                       # cnt[(i, j)] = số hàng trước đó có 1 ở cả cột i và j
        n = len(grid[0])
        for row in grid:
            for i, c1 in enumerate(row):
                if c1:
                    for j in range(i + 1, n):
                        if row[j]:
                            ans += cnt[(i, j)]    # cộng TRƯỚC
                            cnt[(i, j)] += 1      # tăng SAU
        return ans
```

Nhớ ba ý: **hình chữ nhật = hai hàng + hai cột chung**, **hai hàng có `k` cột chung cho `C(k, 2)` hình**, **với bảng băm: cộng trước, tăng sau**.

### Checklist trước khi submit

- [ ] Hiểu rằng bốn góc phải phân biệt: cần hai hàng khác nhau và hai cột khác nhau.
- [ ] Mỗi cặp cột chỉ xét một lần (`j > i`).
- [ ] Cộng `cnt[(i, j)]` **trước** khi tăng nó.
- [ ] Dùng `C(k, 2)`, không phải `k`, khi đếm theo cặp hàng.
- [ ] Ước lượng thời gian: tránh `O(m² n²)` với `200 x 200`.
- [ ] Đã thử `[[1]]`, một hàng toàn `1`, `[[1,1],[1,1]]`, lưới toàn `1`, đường chéo.
- [ ] Time `O(Σ C(k_r, 2))` (≤ `O(m n²)`), Space `O(n²)`.
- [ ] (Premium) đã cân nhắc đối chiếu lại đề gốc nếu có tài khoản.